<a href="https://colab.research.google.com/github/kemmer-ida/Abschlussarbeit_Kemmer/blob/main/Masterarbeit_Auswertung_Kemmer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install odfpy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 717.0/717.0 kB 22.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for odfpy: filename=odfpy-1.4.1-py2.py3-none-any.whl size=160673 sha256=68cb2e4f0fa9a905aa69c474db2ac23194b00cc5628a5df52ffdac0a91b2fdd7
  Stored in directory: /root/.cache/pip/wheels/36/5d/63/8243a7ee78fff0f944d638fd0e66d7278888f5e2285d7346b6
Successfully built odfpy


In [ ]:
!pip install scienceplots

In [ ]:
!pip install scikit_posthocs

In [ ]:
import pandas as pd
from pandas import DataFrame
import numpy as np
from numpy.typing import NDArray
from typing import Literal
from itertools import batched
from scipy import stats
from scikit_posthocs import posthoc_dunn
import matplotlib.pyplot as plt
import scienceplots
plt.style.use(['science', 'grid', 'no-latex'])

In [ ]:
def calculate_paper_values(df: DataFrame,
                           value: Literal['kraftmaximum', 'laengenaenderung', 'bruchdehnung', 'e-modul', 'e-modul berechnet']) -> NDArray:
    match value:
        case 'kraftmaximum':
            values = df['Kraftmaximum [N]'].to_numpy()
        case 'laengenaenderung':
            values = df['Längenänderung [mm]'].to_numpy()
        case 'bruchdehnung':
            values = df['Bruchdehnung [%]'].to_numpy()
        case 'e-modul':
            values = df['E-Modul [N/mm^2]'].to_numpy()
        case 'e-modul berechnet':
            values = df['E-Modul berechnet [N/mm^2]'].to_numpy()
        case _:
            raise ValueError(f'Unknown value: {value}')

    blattgewichte = df["Blattgewicht [g]"].to_numpy()

    paper_values = np.array([])
    for value_pair, (blattgewicht, _) in zip(batched(values, 2), batched(blattgewichte, 2)):
        paper_values = np.append(paper_values, np.mean(value_pair) / blattgewicht)

    return paper_values

def grubbs_outlier_test(data, alpha=0.05):
    if len(data) < 3:
        raise ValueError("Grubbs' Test benötigt mindestens 3 Datenpunkte.")

    data = np.array(data)
    mean = np.mean(data)
    std_dev = np.std(data, ddof=1)

    n = len(data)
    max_val = np.max(data)
    min_val = np.min(data)

    g_max = (max_val - mean) / std_dev
    g_min = (mean - min_val) / std_dev

    g = max(g_max, g_min)

    t_crit = stats.t.ppf(1 - alpha / (2 * n), n - 2)
    critical_value = ((n - 1) / np.sqrt(n)) * np.sqrt((t_crit ** 2) / ((n - 2) + t_crit ** 2))

    outlier_value = max_val if g == g_max else min_val

    return (outlier_value, g, critical_value)

In [ ]:
# Das Hauptdataframe laden
df = pd.read_excel("/content/Auswertung.ods").drop("Bemerkung", axis=1)

# Das DataFrame in die Biomassen aufteilen
blanko_df = df.where(df["Biomasse"] == "n.z.").dropna().reset_index()
schilf_df = df.where(df["Biomasse"] == "Schilf").dropna().reset_index()
rappen_df = df.where(df["Biomasse"] == "Rappen").dropna().reset_index()
trester_df = df.where(df["Biomasse"] == "Trester").dropna().reset_index()

# Die Biomassen in die Katalysatoren aufteilen
schilf_roh_df = schilf_df.where(schilf_df["Katalysator"] == "roh Schilf").dropna().reset_index()
schilf_ohne_df = schilf_df.where(schilf_df["Katalysator"] == "ohne").dropna().reset_index()
schilf_basisch_df = schilf_df.where(schilf_df["Katalysator"] == "basisch").dropna().reset_index()
schilf_sauer_df = schilf_df.where(schilf_df["Katalysator"] == "sauer").dropna().reset_index()

rappen_roh_df = rappen_df.where(rappen_df["Katalysator"] == "roh Rappen").dropna().reset_index()
rappen_ohne_df = rappen_df.where(rappen_df["Katalysator"] == "ohne").dropna().reset_index()
rappen_basisch_df = rappen_df.where(rappen_df["Katalysator"] == "basisch").dropna().reset_index()
rappen_sauer_df = rappen_df.where(rappen_df["Katalysator"] == "sauer").dropna().reset_index()

trester_roh_df = trester_df.where(trester_df["Katalysator"] == "roh Trester").dropna().reset_index()
trester_ohne_df = trester_df.where(trester_df["Katalysator"] == "ohne").dropna().reset_index()
trester_basisch_df = trester_df.where(trester_df["Katalysator"] == "basisch").dropna().reset_index()
trester_sauer_df = trester_df.where(trester_df["Katalysator"] == "sauer").dropna().reset_index()

# Katalysatorn in Aliquote aufteilen
schilf_ohne_1_df = schilf_ohne_df.where(schilf_ohne_df["Aufschlussnr."] == 1).dropna().reset_index(drop=True)
schilf_ohne_2_df = schilf_ohne_df.where(schilf_ohne_df["Aufschlussnr."] == 2).dropna().reset_index(drop=True)
schilf_ohne_3_df = schilf_ohne_df.where(schilf_ohne_df["Aufschlussnr."] == 3).dropna().reset_index(drop=True)

schilf_basisch_1_df = schilf_basisch_df.where(schilf_basisch_df["Aufschlussnr."] == "1B").dropna().reset_index(drop=True)
schilf_basisch_2_df = schilf_basisch_df.where(schilf_basisch_df["Aufschlussnr."] == "2B").dropna().reset_index(drop=True)
schilf_basisch_3_df = schilf_basisch_df.where(schilf_basisch_df["Aufschlussnr."] == "3B").dropna().reset_index(drop=True)

schilf_sauer_1_df = schilf_sauer_df.where(schilf_sauer_df["Aufschlussnr."] == "1S").dropna().reset_index(drop=True)
schilf_sauer_2_df = schilf_sauer_df.where(schilf_sauer_df["Aufschlussnr."] == "2S").dropna().reset_index(drop=True)
schilf_sauer_3_df = schilf_sauer_df.where(schilf_sauer_df["Aufschlussnr."] == "3S").dropna().reset_index(drop=True)

rappen_ohne_1_df = rappen_ohne_df.where(rappen_ohne_df["Aufschlussnr."] == 4).dropna().reset_index(drop=True)
rappen_ohne_2_df = rappen_ohne_df.where(rappen_ohne_df["Aufschlussnr."] == 5).dropna().reset_index(drop=True)
rappen_ohne_3_df = rappen_ohne_df.where(rappen_ohne_df["Aufschlussnr."] == 6).dropna().reset_index(drop=True)

rappen_basisch_1_df = rappen_basisch_df.where(rappen_basisch_df["Aufschlussnr."] == "4B").dropna().reset_index(drop=True)
rappen_basisch_2_df = rappen_basisch_df.where(rappen_basisch_df["Aufschlussnr."] == "5B").dropna().reset_index(drop=True)
rappen_basisch_3_df = rappen_basisch_df.where(rappen_basisch_df["Aufschlussnr."] == "6B").dropna().reset_index(drop=True)

rappen_sauer_1_df = rappen_sauer_df.where(rappen_sauer_df["Aufschlussnr."] == "4S").dropna().reset_index(drop=True)
rappen_sauer_2_df = rappen_sauer_df.where(rappen_sauer_df["Aufschlussnr."] == "5S").dropna().reset_index(drop=True)
rappen_sauer_3_df = rappen_sauer_df.where(rappen_sauer_df["Aufschlussnr."] == "6S").dropna().reset_index(drop=True)

trester_ohne_1_df = trester_ohne_df.where(trester_ohne_df["Aufschlussnr."] == 7).dropna().reset_index(drop=True)
trester_ohne_2_df = trester_ohne_df.where(trester_ohne_df["Aufschlussnr."] == 8).dropna().reset_index(drop=True)
trester_ohne_3_df = trester_ohne_df.where(trester_ohne_df["Aufschlussnr."] == 9).dropna().reset_index(drop=True)

trester_basisch_1_df = trester_basisch_df.where(trester_basisch_df["Aufschlussnr."] == "7B").dropna().reset_index(drop=True)
trester_basisch_2_df = trester_basisch_df.where(trester_basisch_df["Aufschlussnr."] == "8B").dropna().reset_index(drop=True)
trester_basisch_3_df = trester_basisch_df.where(trester_basisch_df["Aufschlussnr."] == "9B").dropna().reset_index(drop=True)

trester_sauer_1_df = trester_sauer_df.where(trester_sauer_df["Aufschlussnr."] == "7S").dropna().reset_index(drop=True)
trester_sauer_2_df = trester_sauer_df.where(trester_sauer_df["Aufschlussnr."] == "8S").dropna().reset_index(drop=True)
trester_sauer_3_df = trester_sauer_df.where(trester_sauer_df["Aufschlussnr."] == "9S").dropna().reset_index(drop=True)

schilf_aliquotes = [schilf_roh_df, schilf_ohne_1_df, schilf_ohne_2_df, schilf_ohne_3_df, schilf_basisch_1_df, schilf_basisch_2_df, schilf_basisch_3_df, schilf_sauer_1_df, schilf_sauer_2_df, schilf_sauer_3_df]
rappen_aliquotes = [rappen_roh_df, rappen_ohne_1_df, rappen_ohne_2_df, rappen_ohne_3_df, rappen_basisch_1_df, rappen_basisch_2_df, rappen_basisch_3_df, rappen_sauer_1_df, rappen_sauer_2_df, rappen_sauer_3_df]
trester_aliquotes = [trester_roh_df, trester_ohne_1_df, trester_ohne_2_df, trester_ohne_3_df, trester_basisch_1_df, trester_basisch_2_df, trester_basisch_3_df, trester_sauer_1_df, trester_sauer_2_df, trester_sauer_3_df]
all_aliquotes = [blanko_df,] + schilf_aliquotes + rappen_aliquotes + trester_aliquotes

FileNotFoundError: [Errno 2] No such file or directory: '/content/Auswertung.ods'

# Normalverteilung

Testet alle Kennwerte der Aliquote auf ihre Normalverteilung mit einem Shapiro-Wilks-Test. Wenn der p-Wert unter 0.05 liegt, kann die Annahme einer Normalverteilung der Daten <ins>nicht</ins> gehalten werden.

In [ ]:
for kennwert in ['kraftmaximum', 'laengenaenderung', 'bruchdehnung', 'e-modul', 'e-modul berechnet']:
    print(kennwert.upper())
    for dataframe in all_aliquotes:
        values = calculate_paper_values(dataframe, kennwert)
        result = stats.shapiro(values)
        if result.pvalue < 0.05:
            s = " HIT!"
        else:
            s = ""
        print(f"{dataframe["Biomasse"][0]} {dataframe["Katalysator"][0]} {dataframe["Aufschlussnr."][0]}: {result.pvalue:.3f}{s}")
    print("\n\n\n")

# Ausreißertest

Testet alle Kennwerte in allen Aliquoten auf Ausreißer nach Grubbs. Wenn ein Ausreißer entdeckt worden ist, wird dieser Wert ausgegeben. Es ist wichtig die Treffer mit den Ergebnissen des Tests auf Normalverteilung zu vergleichen, da der Test keine Aussagekraft hat bei nicht-normalverteilten Daten.

In [ ]:
for kennwert in ['kraftmaximum', 'laengenaenderung', 'bruchdehnung', 'e-modul', 'e-modul berechnet']:
    print(kennwert.upper())
    for dataframe in all_aliquotes:
        values = calculate_paper_values(dataframe, kennwert)
        result = grubbs_outlier_test(values)
        if result[1] > result[2]:
            print(f"{dataframe["Biomasse"][0]} {dataframe["Katalysator"][0]} {dataframe["Aufschlussnr."][0]}: {result[0]:.2f}{s}")
    print("\n\n\n")

# Kruskal-Wallis-Test

In [ ]:
blanko = [blanko_df,]
schilf_roh = [schilf_roh_df,]
rappen_roh = [rappen_roh_df,]
trester_roh = [trester_roh_df,]
schilf_ohne = [schilf_ohne_1_df, schilf_ohne_2_df, schilf_ohne_3_df]
rappen_ohne = [rappen_ohne_1_df, rappen_ohne_2_df, rappen_ohne_3_df]
trester_ohne = [trester_ohne_1_df, trester_ohne_2_df, trester_ohne_3_df]
schilf_basisch = [schilf_basisch_1_df, schilf_basisch_2_df, schilf_basisch_3_df]
rappen_basisch = [rappen_basisch_1_df, rappen_basisch_2_df, rappen_basisch_3_df]
trester_basisch = [trester_basisch_1_df, trester_basisch_2_df, trester_basisch_3_df]
schilf_sauer = [schilf_sauer_1_df, schilf_ohne_2_df, schilf_sauer_3_df]
rappen_sauer = [rappen_sauer_1_df, rappen_sauer_2_df, rappen_sauer_3_df]
trester_sauer = [trester_sauer_1_df, trester_sauer_2_df, trester_sauer_3_df]

groups = [blanko,
          schilf_roh, schilf_ohne, schilf_basisch, schilf_sauer,
          rappen_roh, rappen_ohne, rappen_basisch, rappen_sauer,
          trester_roh, trester_ohne, trester_basisch, trester_sauer]

dunn_results = {}

for kennwert in ['kraftmaximum', 'laengenaenderung', 'bruchdehnung', 'e-modul', 'e-modul berechnet']:
    print(kennwert.upper())

    blanko_median = []
    schilf_roh_median = []
    rappen_roh_median = []
    trester_roh_median = []
    schilf_ohne_median = []
    rappen_ohne_median = []
    trester_ohne_median = []
    schilf_basisch_median = []
    rappen_basisch_median = []
    trester_basisch_median = []
    schilf_sauer_median = []
    rappen_sauer_median = []
    trester_sauer_median = []

    # result_groups = [blanko_median, schilf_roh_median, rappen_roh_median,
    #                 trester_roh_median, schilf_ohne_median, rappen_ohne_median,
    #                trester_ohne_median, schilf_basisch_median,
    #                rappen_basisch_median, trester_basisch_median,
    #                 schilf_sauer_median, rappen_sauer_median, trester_sauer_median]

    result_groups = [blanko_median,
                      schilf_roh_median, schilf_ohne_median, schilf_basisch_median, schilf_sauer_median,
                      rappen_roh_median, rappen_ohne_median, rappen_basisch_median, rappen_sauer_median,
                      trester_roh_median, trester_ohne_median, trester_basisch_median, trester_sauer_median]

    for group, result_group in zip(groups, result_groups):
        for dataframe in group:
            values = calculate_paper_values(dataframe, kennwert)
            result_group.append(np.median(values))

    result = stats.kruskal(*result_groups)
    print(f"pvalue: {result.pvalue}\n")

    # dunn_results[kennwert] = posthoc_dunn(result_groups, p_adjust='bonferroni')
    dunn_results[kennwert] = posthoc_dunn(result_groups, p_adjust='fdr_bh')

In [ ]:
labels = ["B", "SR", "SO", "SB", "SS", "RR", "RO", "RB", "RS", "TR", "TO", "TB", "TS"]

fig, ax = plt.subplots(3, 2, dpi=200, layout='tight', figsize=(10, 15))

ax[0, 0].set_title("Kraftmaximum")
ax[0, 0].imshow(dunn_results['kraftmaximum'], cmap="cividis_r", origin="lower")
ax[0, 0].set_xticks(list(range(13)), labels)
ax[0, 0].set_yticks(list(range(13)), labels)

for i in range(13):
    for j in range(13):
        ax[0, 0].text(j, i, str(round(dunn_results['kraftmaximum'].iloc[i, j], 2)), ha='center', va='center', color='k')

ax[0, 1].set_title("Laengenaenderung")
ax[0, 1].imshow(dunn_results['laengenaenderung'], cmap="cividis_r", origin="lower")
ax[0, 1].set_xticks(list(range(13)), labels)
ax[0, 1].set_yticks(list(range(13)), labels)

for i in range(13):
    for j in range(13):
        ax[0, 1].text(j, i, str(round(dunn_results['laengenaenderung'].iloc[i, j], 2)), ha='center', va='center', color='k')

ax[1, 0].set_title("Bruchdehnung")
ax[1, 0].imshow(dunn_results['bruchdehnung'], cmap="cividis_r", origin="lower")
ax[1, 0].set_xticks(list(range(13)), labels)
ax[1, 0].set_yticks(list(range(13)), labels)

for i in range(13):
    for j in range(13):
        ax[1, 0].text(j, i, str(round(dunn_results['bruchdehnung'].iloc[i, j], 2)), ha='center', va='center', color='k')

ax[1, 1].set_title("E-Modul nicht-berechnet")
ax[1, 1].imshow(dunn_results['e-modul'], cmap="cividis_r", origin="lower")
ax[1, 1].set_xticks(list(range(13)), labels)
ax[1, 1].set_yticks(list(range(13)), labels)

for i in range(13):
    for j in range(13):
        ax[1, 1].text(j, i, str(round(dunn_results['e-modul'].iloc[i, j], 2)), ha='center', va='center', color='k')

ax[2, 0].set_title("E-Modul")
ax[2, 0].imshow(dunn_results['e-modul berechnet'], cmap="cividis_r", origin="lower")
ax[2, 0].set_xticks(list(range(13)), labels)
ax[2, 0].set_yticks(list(range(13)), labels)

for i in range(13):
    for j in range(13):
        ax[2, 0].text(j, i, str(round(dunn_results['e-modul berechnet'].iloc[i, j], 2)), ha='center', va='center', color='k')

ax[2, 1].axis('off')
plt.show()

In [ ]:
fig, ax = plt.subplots(5, 1, dpi=200, layout='tight', figsize=(10, 15))
ax = ax.flatten()

labels = ["B", "SR", "SO1", "SO2", "SO3", "SB1", "SB2", "SB3", "SS1", "SS2", "SS3",
          "RR", "RO1", "RO2", "RO3", "RB1", "RB2", "RB3", "RS1", "RS2", "RS3",
          "TR", "TO1", "TO2", "TO3", "TB1", "TB2", "TB3", "TS1", "TS2", "TS3"]

for i, kennwert in enumerate(['kraftmaximum', 'laengenaenderung', 'bruchdehnung', 'e-modul', 'e-modul berechnet']):
    all_values = []
    for dataframe in all_aliquotes:
        values = calculate_paper_values(dataframe, kennwert)
        all_values.append(values)

    ax[i].set_title(kennwert)
    ax[i].boxplot(all_values)
    ax[i].set_xticks(list(range(1, len(labels) + 1)), labels)

plt.show()
# [blanko_df,] + schilf_aliquotes + rappen_aliquotes + trester_aliquotes

In [ ]:
import matplotlib.ticker as ticker

fig, ax = plt.subplots(5, 1, dpi=200, layout='tight', figsize=(10, 15))
ax = ax.flatten()

labels = ["B", "SR", "SO", "SB", "SS", "RR", "RO", "RB", "RS", "TR", "TO", "TB", "TS"]
title = ["spezifisches Kraftmaximum", "spezifische Längenänderung", "spezifische Bruchdehnung", "spezifischer E-Modul nicht berechnet", "spezifischer E-Modul"]
units = ["N g$^{-1}$", "% g$^{-1}$", "% g$^{-1}$", "N mm$^{-2}$ g$^{-1}$", "N mm$^{-2}$ g$^{-1}$"]

blanko = [blanko_df,]
schilf_roh = [schilf_roh_df,]
rappen_roh = [rappen_roh_df,]
trester_roh = [trester_roh_df,]
schilf_ohne = [schilf_ohne_1_df, schilf_ohne_2_df, schilf_ohne_3_df]
rappen_ohne = [rappen_ohne_1_df, rappen_ohne_2_df, rappen_ohne_3_df]
trester_ohne = [trester_ohne_1_df, trester_ohne_2_df, trester_ohne_3_df]
schilf_basisch = [schilf_basisch_1_df, schilf_basisch_2_df, schilf_basisch_3_df]
rappen_basisch = [rappen_basisch_1_df, rappen_basisch_2_df, rappen_basisch_3_df]
trester_basisch = [trester_basisch_1_df, trester_basisch_2_df, trester_basisch_3_df]
schilf_sauer = [schilf_sauer_1_df, schilf_ohne_2_df, schilf_sauer_3_df]
rappen_sauer = [rappen_sauer_1_df, rappen_sauer_2_df, rappen_sauer_3_df]
trester_sauer = [trester_sauer_1_df, trester_sauer_2_df, trester_sauer_3_df]

groups = [blanko,
          schilf_roh, schilf_ohne, schilf_basisch, schilf_sauer,
          rappen_roh, rappen_ohne, rappen_basisch, rappen_sauer,
          trester_roh, trester_ohne, trester_basisch, trester_sauer]

for i, kennwert in enumerate(['kraftmaximum', 'laengenaenderung', 'bruchdehnung', 'e-modul', 'e-modul berechnet']):

    blanko_median = []
    schilf_roh_median = []
    rappen_roh_median = []
    trester_roh_median = []
    schilf_ohne_median = []
    rappen_ohne_median = []
    trester_ohne_median = []
    schilf_basisch_median = []
    rappen_basisch_median = []
    trester_basisch_median = []
    schilf_sauer_median = []
    rappen_sauer_median = []
    trester_sauer_median = []

   # result_groups = [blanko_median, schilf_roh_median, rappen_roh_median,
    #                 trester_roh_median, schilf_ohne_median, rappen_ohne_median,
     #                trester_ohne_median, schilf_basisch_median,
      #               rappen_basisch_median, trester_basisch_median,
       #              schilf_sauer_median, rappen_sauer_median, trester_sauer_median]

    result_groups = [blanko_median,
                      schilf_roh_median, schilf_ohne_median, schilf_basisch_median, schilf_sauer_median,
                      rappen_roh_median, rappen_ohne_median, rappen_basisch_median, rappen_sauer_median,
                      trester_roh_median, trester_ohne_median, trester_basisch_median, trester_sauer_median]

    for group, result_group in zip(groups, result_groups):
        for dataframe in group:
            values = calculate_paper_values(dataframe, kennwert)
            result_group.append(np.median(values))

    ax[i].yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f"{x:.2f}".replace('.', ',')))

    ax[i].set_title(title[i], fontsize=14)
    ax[i].set_ylabel(f"{title[i]} [{units[i]}]", fontsize=11)
    ax[i].boxplot(result_groups)
    ax[i].set_xticks(list(range(1, len(labels) + 1)), labels, fontsize=11)

plt.show()

In [ ]:
# Deine Daten und Gruppen (wie im Originalcode)
labels = ["B", "SR", "RR", "TR", "SO", "RO", "TO", "SB", "RB", "TB", "SS", "RS", "TS"]
title = ["Kraftmaximum", "Laengenaenderung", "Bruchdehnung", "E-Modul", "E-Modul berechnet"]

groups = [
    [blanko_df],
    [schilf_roh_df],
    [rappen_roh_df],
    [trester_roh_df],
    [schilf_ohne_1_df, schilf_ohne_2_df, schilf_ohne_3_df],
    [rappen_ohne_1_df, rappen_ohne_2_df, rappen_ohne_3_df],
    [trester_ohne_1_df, trester_ohne_2_df, trester_ohne_3_df],
    [schilf_basisch_1_df, schilf_basisch_2_df, schilf_basisch_3_df],
    [rappen_basisch_1_df, rappen_basisch_2_df, rappen_basisch_3_df],
    [trester_basisch_1_df, trester_basisch_2_df, trester_basisch_3_df],
    [schilf_sauer_1_df, schilf_ohne_2_df, schilf_sauer_3_df],
    [rappen_sauer_1_df, rappen_sauer_2_df, rappen_sauer_3_df],
    [trester_sauer_1_df, trester_sauer_2_df, trester_sauer_3_df]
]

# Funktion zur Berechnung der Mediane und MADs
def calculate_median_mad(groups, kennwert):
    medians = []
    mads = []
    for group in groups:
        values = []
        for dataframe in group:
            values.extend(calculate_paper_values(dataframe, kennwert))
        median = np.median(values)
        mad = stats.median_abs_deviation(values, scale='normal')
        medians.append(median)
        mads.append(mad)
    return medians, mads

# Berechne und gebe die Mediane und MADs für jeden Kennwert aus
for i, kennwert in enumerate(['kraftmaximum', 'laengenaenderung', 'bruchdehnung', 'e-modul', 'e-modul berechnet']):
    medians, mads = calculate_median_mad(groups, kennwert)
    print(f"\n--- {title[i]} ---")
    for j, label in enumerate(labels):
        print(f"{label}: Median = {medians[j]:.2f}, MAD = {mads[j]:.2f}")

visuelle Darstellung der Papierdicken

In [ ]:
# Mittelwert und Standardabweichung der experimentell ermittelten Papierdicken

# Daten
data = {
    "Probe": ["B", "SR", "SO", "SB", "SS", "RR", "RO", "RB", "RS", "TR", "TO", "TB", "TS"],
    "Mittelwert [μm]": [300.0, 436.6, 393.6, 315.7, 347.0, 447.8, 409.5, 335.1, 410.1, 387.1, 479.2, 378.6, 465.1],
    "Standardabweichung [μm]": [29.18, 28.58, 25.25, 27.81, 32.59, 24.79, 40.03, 25.51, 29.26, 35.51, 39.53, 35.25, 40.76]
}

# DataFrame erstellen
df = pd.DataFrame(data)

# Diagramm erstellen
plt.figure(figsize=(12, 6))
plt.errorbar(
    x=df["Probe"],
    y=df["Mittelwert [μm]"],
    yerr=df["Standardabweichung [μm]"],
    fmt='_',
    color='lightgreen', # Farbe der Marker
    ecolor='black',  # Farbe der Fehlerbalken
    capsize=5,     # Länge der Querstriche an den Fehlerbalken
    capthick=2,    # Dicke der Querstriche
    elinewidth=1.5,  # Dicke der Fehlerbalken
    markeredgewidth=1.5,
    markersize=8
)

# Beschriftungen und Titel
plt.ylabel("Dicke [μm]", fontsize=11)
plt.xlabel("", fontsize=11)
plt.title("Mittelwert der Papierdicke", fontsize=16)
plt.grid(axis='y', linestyle='--', alpha=0.7)

# Y-Achse anpassen
plt.ylim(200, df["Mittelwert [μm]"].max() + df["Standardabweichung [μm]"].max() + 20)

# Diagramm anzeigen
plt.tight_layout()
plt.show()

t-Test Fibertherm Daten

In [ ]:
# --- DATEN ADF ---
daten = {
    "Schilf": {
        "Kontrolle": 68.9264,  # Einzelwert
        "O": [76.9351, 75.7484, 73.4765],
        "B": [62.1274, 59.5019, 61.9945],
        "S": [86.5725, 81.5174, 84.5947]
    },
    "Rappen": {
        "Kontrolle": 71.9673,
        "O": [79.9485, 75.3867, 75.5915],
        "B": [63.9311, 66.9303, 62.0770],
        "S": [75.3882, 74.3677, 73.8089]
    },
    "Trester": {
        "Kontrolle": 57.0248,
        "O": [85.7173, 84.9453, 90.4144],
        "B": [65.6211, 61.1179, 67.9041],
        "S": [85.8682, 87.3275, 93.4448]
    }
}

# --- FUNKTION FÜR EINSTICHPROBEN-T-TEST ---
def t_test_1samp(kontrollwert, behandlung, name_kontrolle, name_behandlung):
    t_stat, p_wert = stats.ttest_1samp(behandlung, popmean=kontrollwert)
    print(f"\n--- {name_kontrolle} vs. {name_behandlung} ---")
    print(f"Kontrollwert: {kontrollwert:.2f}")
    print(f"Mittelwert {name_behandlung}: {np.mean(behandlung):.2f} ± {np.std(behandlung):.4f}")
    print(f"t-Statistik: {t_stat:.3f}, p-Wert: {p_wert:.4f}")
    if p_wert < 0.05:
        print("→ **Signifikanter Unterschied (p < 0.05)**")
    else:
        print("→ Kein signifikanter Unterschied (p ≥ 0.05)")

# --- T-TESTS DURCHFÜHREN ---
for biomasse, werte in daten.items():
    kontrollwert = werte["Kontrolle"]
    for variant in ["O", "B", "S"]:
        t_test_1samp(
            kontrollwert,
            werte[variant],
            f"{biomasse} (Kontrolle)",
            f"{biomasse} {variant}"
        )


--- Schilf (Kontrolle) vs. Schilf O ---
Kontrollwert: 68.93
Mittelwert Schilf O: 75.39 ± 1.4349
t-Statistik: 6.367, p-Wert: 0.0238
→ **Signifikanter Unterschied (p < 0.05)**

--- Schilf (Kontrolle) vs. Schilf B ---
Kontrollwert: 68.93
Mittelwert Schilf B: 61.21 ± 1.2076
t-Statistik: -9.039, p-Wert: 0.0120
→ **Signifikanter Unterschied (p < 0.05)**

--- Schilf (Kontrolle) vs. Schilf S ---
Kontrollwert: 68.93
Mittelwert Schilf S: 84.23 ± 2.0799
t-Statistik: 10.404, p-Wert: 0.0091
→ **Signifikanter Unterschied (p < 0.05)**

--- Rappen (Kontrolle) vs. Rappen O ---
Kontrollwert: 71.97
Mittelwert Rappen O: 76.98 ± 2.1038
t-Statistik: 3.367, p-Wert: 0.0780
→ Kein signifikanter Unterschied (p ≥ 0.05)

--- Rappen (Kontrolle) vs. Rappen B ---
Kontrollwert: 71.97
Mittelwert Rappen B: 64.31 ± 1.9997
t-Statistik: -5.413, p-Wert: 0.0325
→ **Signifikanter Unterschied (p < 0.05)**

--- Rappen (Kontrolle) vs. Rappen S ---
Kontrollwert: 71.97
Mittelwert Rappen S: 74.52 ± 0.6539
t-Statistik: 5.525, p-We

In [ ]:
# --- ADL-DATEN ---
daten = {
    "Schilf": {
        "Kontrolle": 12.8382,
        "O": [10.9952, 0.6592, 8.3133],
        "B": [0.3208, 2.2799, 0.3265],
        "S": [0.2653, 0.8004, 0.4589]
    },
    "Rappen": {
        "Kontrolle": 23.6935,
        "O": [6.4468, 3.7257, 25.2523],
        "B": [0.5243, 0.7268, 5.8985],
        "S": [3.3545, 17.3711, 22.0036]
    },
    "Trester": {
        "Kontrolle": 43.4946,
        "O": [71.6305, 66.3996, 76.1760],
        "B": [49.7841, 44.2445, 51.1937],
        "S": [66.0939, 73.3481, 81.5602]
    }
}

# --- FUNKTION FÜR EINSTICHPROBEN-T-TEST ---
def t_test_1samp(kontrollwert, behandlung, name_kontrolle, name_behandlung):
    t_stat, p_wert = stats.ttest_1samp(behandlung, popmean=kontrollwert)
    print(f"\n--- {name_kontrolle} vs. {name_behandlung} ---")
    print(f"Kontrollwert: {kontrollwert:.2f}")
    print(f"Mittelwert {name_behandlung}: {np.mean(behandlung):.2f} ± {np.std(behandlung):.4f}")
    print(f"t-Statistik: {t_stat:.3f}, p-Wert: {p_wert:.4f}")
    if p_wert < 0.05:
        print("→ **Signifikanter Unterschied (p < 0.05)**")
    else:
        print("→ Kein signifikanter Unterschied (p ≥ 0.05)")

# --- T-TESTS DURCHFÜHREN ---
for biomasse, werte in daten.items():
    kontrollwert = werte["Kontrolle"]
    for variant in ["O", "B", "S"]:
        t_test_1samp(
            kontrollwert,
            werte[variant],
            f"{biomasse} (Kontrolle)",
            f"{biomasse} {variant}"
        )


--- Schilf (Kontrolle) vs. Schilf O ---
Kontrollwert: 12.84
Mittelwert Schilf O: 6.66 ± 4.3794
t-Statistik: -1.996, p-Wert: 0.1840
→ Kein signifikanter Unterschied (p ≥ 0.05)

--- Schilf (Kontrolle) vs. Schilf B ---
Kontrollwert: 12.84
Mittelwert Schilf B: 0.98 ± 0.9222
t-Statistik: -18.192, p-Wert: 0.0030
→ **Signifikanter Unterschied (p < 0.05)**

--- Schilf (Kontrolle) vs. Schilf S ---
Kontrollwert: 12.84
Mittelwert Schilf S: 0.51 ± 0.2212
t-Statistik: -78.824, p-Wert: 0.0002
→ **Signifikanter Unterschied (p < 0.05)**

--- Rappen (Kontrolle) vs. Rappen O ---
Kontrollwert: 23.69
Mittelwert Rappen O: 11.81 ± 9.5711
t-Statistik: -1.756, p-Wert: 0.2211
→ Kein signifikanter Unterschied (p ≥ 0.05)

--- Rappen (Kontrolle) vs. Rappen B ---
Kontrollwert: 23.69
Mittelwert Rappen B: 2.38 ± 2.4871
t-Statistik: -12.118, p-Wert: 0.0067
→ **Signifikanter Unterschied (p < 0.05)**

--- Rappen (Kontrolle) vs. Rappen S ---
Kontrollwert: 23.69
Mittelwert Rappen S: 14.24 ± 7.9282
t-Statistik: -1.686, p

In [ ]:
# --- ASCHEGEHALT-DATEN ---
daten = {
    "Schilf": {
        "Kontrolle": 1.2890,
        "O": [2.5117, 0.2065, 1.3035],
        "B": [0.5003, 0.9474, 0.1086],
        "S": [0.1764, 0.2800, 0.1486]
    },
    "Rappen": {
        "Kontrolle": 0.7698,
        "O": [0.1879, 0.1676, 0.7152],
        "B": [0.0834, 0.0592, 0.2665],
        "S": [0.0297, 0.5387, 0.5875]
    },
    "Trester": {
        "Kontrolle": 0.6316,
        "O": [10.8842, 5.5784, 11.5812],
        "B": [7.1921, 4.1859, 5.0943],
        "S": [17.3288, 26.5008, 18.9581]
    }
}

# --- FUNKTION FÜR EINSTICHPROBEN-T-TEST ---
def t_test_1samp(kontrollwert, behandlung, name_kontrolle, name_behandlung):
    t_stat, p_wert = stats.ttest_1samp(behandlung, popmean=kontrollwert)
    print(f"\n--- {name_kontrolle} vs. {name_behandlung} ---")
    print(f"Kontrollwert: {kontrollwert:.2f}")
    print(f"Mittelwert {name_behandlung}: {np.mean(behandlung):.2f} ± {np.std(behandlung):.4f}")
    print(f"t-Statistik: {t_stat:.3f}, p-Wert: {p_wert:.4f}")
    if p_wert < 0.05:
        print("→ **Signifikanter Unterschied (p < 0.05)**")
    else:
        print("→ Kein signifikanter Unterschied (p ≥ 0.05)")

# --- T-TESTS DURCHFÜHREN ---
for biomasse, werte in daten.items():
    kontrollwert = werte["Kontrolle"]
    for variant in ["O", "B", "S"]:
        t_test_1samp(
            kontrollwert,
            werte[variant],
            f"{biomasse} (Kontrolle)",
            f"{biomasse} {variant}"
        )


--- Schilf (Kontrolle) vs. Schilf O ---
Kontrollwert: 1.29
Mittelwert Schilf O: 1.34 ± 0.9415
t-Statistik: 0.077, p-Wert: 0.9453
→ Kein signifikanter Unterschied (p ≥ 0.05)

--- Schilf (Kontrolle) vs. Schilf B ---
Kontrollwert: 1.29
Mittelwert Schilf B: 0.52 ± 0.3427
t-Statistik: -3.179, p-Wert: 0.0863
→ Kein signifikanter Unterschied (p ≥ 0.05)

--- Schilf (Kontrolle) vs. Schilf S ---
Kontrollwert: 1.29
Mittelwert Schilf S: 0.20 ± 0.0565
t-Statistik: -27.197, p-Wert: 0.0013
→ **Signifikanter Unterschied (p < 0.05)**

--- Rappen (Kontrolle) vs. Rappen O ---
Kontrollwert: 0.77
Mittelwert Rappen O: 0.36 ± 0.2535
t-Statistik: -2.304, p-Wert: 0.1478
→ Kein signifikanter Unterschied (p ≥ 0.05)

--- Rappen (Kontrolle) vs. Rappen B ---
Kontrollwert: 0.77
Mittelwert Rappen B: 0.14 ± 0.0925
t-Statistik: -9.680, p-Wert: 0.0105
→ **Signifikanter Unterschied (p < 0.05)**

--- Rappen (Kontrolle) vs. Rappen S ---
Kontrollwert: 0.77
Mittelwert Rappen S: 0.39 ± 0.2522
t-Statistik: -2.156, p-Wert: 0.1

visuelle Auswertung der Fibertherm Daten

In [ ]:
# Daten
biomassen = ["SR", "SO", "SB", "SS", "RR", "RO", "RB", "RS", "TR", "TO", "TB", "TS"]

# ADF-Daten (Mittelwert und Standardabweichung)
adf_mittelwert = [68.9, 75.4, 61.2, 84.2, 72.0, 77.0, 64.3, 74.5, 57.0, 87.0, 64.9, 88.9]
adf_std = [0, 1.8, 1.5, 2.5, 0, 2.6, 2.4, 0.8, 0, 3.0, 3.5, 4.0]  # 0, wenn keine Standardabweichung angegeben

# ADL-Daten (Mittelwert und Standardabweichung)
adl_mittelwert = [12.8, 6.7, 1.0, 0.5, 23.7, 11.8, 2.4, 14.2, 43.5, 71.4, 48.4, 73.7]
adl_std = [0, 5.4, 1.1, 0.3, 0, 11.7, 3.0, 9.7, 0, 4.9, 3.7, 7.7]  # 0, wenn keine Standardabweichung angegeben

# Aschegehalt-Daten (Mittelwert und Standardabweichung)
asche_mittelwert = [1.3, 1.3, 0.5, 0.2, 0.8, 0.4, 0.1, 0.4, 0.6, 9.3, 5.5, 20.9]
asche_std = [0, 1.2, 0.4, 0.1, 0, 0.3, 0.1, 0.3, 0, 3.3, 1.5, 4.9]  # 0, wenn keine Standardabweichung angegeben

# Funktion zur Berechnung des Variationskoeffizienten (in %)
def berechne_vk(mittelwert, std):
    vk = []
    for m, s in zip(mittelwert, std):
        if m == 0:
            vk.append(0)  # Vermeide Division durch Null
        else:
            vk.append((s / m) * 100 if s != 0 else 0)
    return vk

# Variationskoeffizienten berechnen
adf_vk = berechne_vk(adf_mittelwert, adf_std)
adl_vk = berechne_vk(adl_mittelwert, adl_std)
asche_vk = berechne_vk(asche_mittelwert, asche_std)

# Ergebnisse als Tabelle ausgeben
print("Variationskoeffizienten (VK) in %:")
print(f"{'Biomasse':<6} | {'ADF VK':<8} | {'ADL VK':<8} | {'Asche VK':<8}")
print("-" * 40)
for i, bio in enumerate(biomassen):
    print(f"{bio:<6} | {adf_vk[i]:<8.2f} | {adl_vk[i]:<8.2f} | {asche_vk[i]:<8.2f}")

# Funktion zum Erstellen eines Diagramms
def plot_data(biomassen, mittelwert, std, title, ylabel):
    plt.figure(figsize=(12, 6))
    plt.errorbar(
        x=biomassen,
        y=mittelwert,
        yerr=std,
        fmt='_',
        color='green',
        ecolor='black',
        capsize=5,
        capthick=2,
        elinewidth=1.5,
        markeredgewidth=1.5,
        markersize=8
    )

    plt.ylabel(ylabel, fontsize=13)
    plt.title(title, fontsize=16)
    plt.xticks(ha='right', fontsize=13)
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.ylim(0, max(mittelwert) + max(std) + 5)  # Y-Achse anpassen
    plt.tight_layout()
    plt.show()

# Diagramme erstellen
plot_data(biomassen, adf_mittelwert, adf_std, "Mittelwert ADF$_{\mathrm{DM\,basis}}$-Gehalt", "ADF$_{\mathrm{DM\,basis}}$-Gehalt [%]")
plot_data(biomassen, adl_mittelwert, adl_std, "Mittelwert ADL$_{\mathrm{DM\,basis}}$-Gehalt", "ADL$_{\mathrm{DM\,basis}}$-Gehalt [%]")
plot_data(biomassen, asche_mittelwert, asche_std, "Mittelwert Aschegehalt", "Aschegehalt [%]")